In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import pandas as pd
import string
import re
from datasets import load_dataset
from tqdm.auto import tqdm
from notebooks.utils import load_llama, create_prompts_nq, generate_batch

In [3]:
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
DATASET_ID = "florin-hf/nq_open_gold"
BATCH_SIZE = 16

In [4]:
def normalize_answer(s):
    """
    Standard normalization function for QA (SQuAD/NQ).
    """
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def calculate_exact_match(prediction, ground_truths):
    """
    Checks if the normalized prediction matches ANY of the normalized ground truths.
    """
    norm_pred = normalize_answer(prediction)
    norm_truths = [normalize_answer(t) for t in ground_truths]
    return max([1 if norm_pred == nt else 0 for nt in norm_truths])

In [5]:
def create_rag_prompts(tokenizer, questions, contexts):
    """
    Creates RAG prompts injecting the Gold Context.
    """
    prompts = []
    for q, ctx in zip(questions, contexts):
        messages = [
            # Constrain the model to use the context
            {
                "role": "system", 
                "content": "Answer the question using only the provided context. Answer with a short phrase or entity name only."
            },
            {
                "role": "user", 
                "content": f"Context:\n{ctx}\n\nQuestion: {q}"
            },
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)
    return prompts

In [6]:
tokenizer, model = load_llama(MODEL_ID)

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [7]:
ds = load_dataset(DATASET_ID, split="validation")

df = pd.DataFrame({
    'question': ds['question'],
    'answers': ds['answers'], # List of valid answers
    'context': ds['text']     # The Gold Paragraph
})

In [8]:
print(f"\nStarting Comparative Evaluation on {len(df)} samples...")
print(f"Metric: Exact Match (EM)")

all_preds_no_rag = []
all_preds_rag = []

# 3. Batch Processing
for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = df.iloc[i : i+BATCH_SIZE]
    questions = batch['question'].tolist()
    contexts = batch['context'].tolist()
    
    # --- A. NO RAG (Closed Book) ---
    prompts_no_rag = create_prompts_nq(tokenizer, questions, contexts=None)
    batch_preds_no_rag = generate_batch(model, tokenizer, prompts_no_rag)
    
    # Clean up
    cleaned_no_rag = [p.replace("Answer:", "").strip() for p in batch_preds_no_rag]
    all_preds_no_rag.extend(cleaned_no_rag)

    # --- B. WITH RAG (Oracle) ---
    prompts_rag = create_prompts_nq(tokenizer, questions, contexts=contexts)
    batch_preds_rag = generate_batch(model, tokenizer, prompts_rag)
    
    # Clean up
    cleaned_rag = [p.replace("Answer:", "").strip() for p in batch_preds_rag]
    all_preds_rag.extend(cleaned_rag)
    
# 4. Calculate Scores
df['pred_no_rag'] = all_preds_no_rag
df['pred_rag'] = all_preds_rag

# Apply EM metric
df['em_no_rag'] = df.apply(lambda row: calculate_exact_match(row['pred_no_rag'], row['answers']), axis=1)
df['em_rag'] = df.apply(lambda row: calculate_exact_match(row['pred_rag'], row['answers']), axis=1)

score_no_rag = df['em_no_rag'].mean() * 100
score_rag = df['em_rag'].mean() * 100

print("\n" + "="*50)
print(f"FINAL RESULTS")
print("="*50)
print(f"Closed Book EM: {score_no_rag:.2f}%")
print(f"Oracle RAG EM:  {score_rag:.2f}%")
print(f"Improvement:    +{score_rag - score_no_rag:.2f}%")

# Show samples where RAG helped
print("\nSuccess Cases (Where RAG fixed the error):")
# Filter for rows where NoRAG=0 and RAG=1
success_cases = df[(df['em_no_rag'] == 0) & (df['em_rag'] == 1)].sample(min(5, len(df)))
for _, row in success_cases.iterrows():
    print(f"Q: {row['question']}")
    print(f"❌ No RAG: {row['pred_no_rag']}")
    print(f"✅ RAG:    {row['pred_rag']}")
    print(f"Real:      {row['answers']}")
    print("-" * 30)


Starting Comparative Evaluation on 8006 samples...
Metric: Exact Match (EM)


  0%|          | 0/501 [00:00<?, ?it/s]


FINAL RESULTS
Closed Book EM: 6.30%
Oracle RAG EM:  36.75%
Improvement:    +30.45%

Success Cases (Where RAG fixed the error):
Q: who's in george michael's freedom video
❌ No RAG: Wham!
✅ RAG:    Christy Turlington
Real:      ['Linda Evangelista', 'Naomi Campbell', 'Christy Turlington', 'Tatjana Patitz', 'Cindy Crawford']
------------------------------
Q: fatty acids that cannot be synthesized within an organism are referred to as
❌ No RAG: Ketones
✅ RAG:    Non-essential fatty acids
Real:      ['non-essential fatty acids']
------------------------------
Q: who has been considered as universal blood doner person
❌ No RAG: Oscar Pistorius
✅ RAG:    Type O Rh D negative
Real:      ['type O Rh D negative']
------------------------------
Q: the predominant texture in the classical era was
❌ No RAG: Marble
✅ RAG:    Homophonic
Real:      ['homophonic']
------------------------------
Q: who won the national championship in football in 2016
❌ No RAG: Alabama
✅ RAG:    Alabama Crimson Tide
Re